# 09 — Génération, citations et conversation

Explique le budget du prompt, la réponse fidèle aux sources, le cache et le traitement des questions de suivi.

> Vérifiez toujours les drapeaux `RUN_*` avant l'exécution. Pour cette reconstruction propre, les traitements locaux nécessaires des étapes 02 à 07 sont déjà activés ; les collectes réseau et évaluations GPU facultatives restent désactivées.


In [1]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('Données actives :', PROJECT_ROOT / 'data')


Projet : <PROJECT_ROOT>
Python : c:\Users\Ariel\OneDrive - ENSEA\Documents\LLMs\RAG ENSEA\.venv\Scripts\python.exe
Données actives : <PROJECT_ROOT>\data


## Règles de génération

Le générateur répond uniquement avec les passages récupérés. Chaque affirmation factuelle reçoit une référence `[n]`. Il vérifie séparément l'indicateur, la population, la période, l'unité et le statut (réalisé, estimé ou prévu), sans déduire une période manquante depuis une période voisine. Lorsqu'une part explicite a été calculée par le résolveur structuré, il reprend le résultat déterministe et ses deux opérandes compatibles sans refaire le calcul. Une comparaison utilise uniquement la relation explicitement écrite dans une preuve, et les labels compacts comme `3x Kaggle Master` sont conservés mot pour mot au lieu d'être interprétés comme des victoires ou diplômes. Les questions numériques simples disposent d'un budget de sortie plus court afin de réduire les digressions et le temps de réponse. L'historique sert seulement à comprendre une relance ; les anciennes réponses ne deviennent jamais des preuves et leurs anciens numéros de citation sont supprimés.


In [2]:
import pandas as pd
from IPython.display import Markdown, display
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
RUN_GENERATION = True
if RUN_GENERATION:
    from rag_pipeline import RAGPipeline
    pipeline = RAGPipeline(config)
    first_question = "Quels sont les finalistes du concours d'éloquence du CDR de cette année ?"
    first = pipeline.ask(first_question)
    display(Markdown(first.answer))
    history = [
        {'role': 'user', 'content': first_question},
        {'role': 'assistant', 'content': first.answer},
    ]
    followup = pipeline.ask(
        'La liste est-elle exhaustive ? Je veux la liste exhaustive.',
        conversation_history=history,
    )
    display(Markdown(followup.answer))

    # Critère d'acceptation conversationnel : une relance exhaustive ne doit
    # perdre aucun élément déjà documenté ni inventer un membre anonyme.
    import unicodedata

    def normalize_answer(text):
        normalized = unicodedata.normalize('NFKD', text.casefold())
        return ''.join(char for char in normalized if not unicodedata.combining(char))

    expected_finalists = {
        'Aka André': 'aka',
        'Ouattara Marc': 'ouattara',
        'TONATO Akoua Yasmine Corine': 'tonato',
        'Kahi Jean Marcel': 'kahi',
        'Gbossa Jérémie': 'gbossa',
        'Zamblé Lou Chimène': 'zambl',
    }
    first_normalized = normalize_answer(first.answer)
    followup_normalized = normalize_answer(followup.answer)
    validation = pd.DataFrame([
        {
            'finaliste': name,
            'premier_tour': marker in first_normalized,
            'relance_exhaustive': marker in followup_normalized,
        }
        for name, marker in expected_finalists.items()
    ])
    display(validation)
    assert validation['premier_tour'].all(), 'Le premier tour ne contient pas les six finalistes.'
    assert validation['relance_exhaustive'].all(), 'La relance exhaustive a perdu au moins un finaliste.'
    assert 'non nomm' not in followup_normalized, 'Le modèle a inventé un candidat non nommé.'
    assert pipeline.retriever.last_decision.route == 'facebook'
    assert pipeline.last_contextualized and pipeline.last_expanded_retrieval
else:
    print('Génération désactivée. Libérez la VRAM puis passez RUN_GENERATION à True.')


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Les finalistes du Concours ENSEA d’Éloquence identifiés sont Aka André (premier), Ouattara Marc (deuxième), Kahi Jean Marcel (quatrième), Gbossa Jérémie (cinquième), TONATO Akoua Yasmine Corine et Zamblé Lou Chimène (sixième). Ces six candidats se distinguent tous par leur maîtrise oratoire lors des présélections avant la finale prévue pour le 18 avril 2026 où s'est imposée comme gagnante TONATO Corinne [7][1][2][3][4][5][6].

Les publications officielles du concours ENSEA d’Éloquence permettent d'identifier six finalistes annoncés successivement au fil des semaines précédant la finale tenue le 18 avril 2026[7][9][10]. La première sélection a désigné Aka André comme premier finaliste pour son assurance remarquable et sa maîtrise dialectique[1]. Deux jours plus tard, Ouattara Marc s'est imposé en tant que deuxième finaliste grâce à une argumentation captivante qui surprend les jurés[3]. Le quatrième rang est occupé par Kahi Jean Marcel, reconnu pour ses techniques oratoires redoutables et sa gestuelle stratégique[2]. TONATO Akoua Yasmine Corine figure ensuite parmi les nouvelles finalistes présentées après ces deux premiers candidats, incarnant un style posé entre rigueur analytique et finesse verbale[6]. Gbossa Jérémie complète cette série de révélations avant l'apparition de la dernière candidate : Zamblé Lou Chimène, qualifiée explicitement de « sixième et dernière finaliste » dans les communications publiques[5][4]. Bien qu'une publication mentionne également Nankam Astrid Rodirne lors d'élections internes liées au CDR (GDR) [8], aucune source ne confirme ici son statut de candidat ou de finaliste du concours d'Éloquence spécifique.

,finaliste,premier_tour,relance_exhaustive
0,Aka André,True,True
1,Ouattara Marc,True,True
2,TONATO Akoua Yasmine Corine,True,True
3,Kahi Jean Marcel,True,True
4,Gbossa Jérémie,True,True
5,Zamblé Lou Chimène,True,True


In [3]:
if RUN_GENERATION:
    display(pd.Series({
        **pipeline.generator.last_prompt_stats,
        **pipeline.generator.last_generation_stats,
        'route': pipeline.retriever.last_decision.route,
        'contextualized': pipeline.last_contextualized,
        'expanded_retrieval': pipeline.last_expanded_retrieval,
        'citations': len(followup.citations),
    }).to_frame('valeur'))
    display(pd.DataFrame([citation.__dict__ for citation in followup.citations]))
    display(pd.DataFrame([
        {
            'rang': index,
            'titre': chunk.title,
            'dans_le_prompt': index in pipeline.generator.prompt_builder.last_passages,
            'couverture_obligatoire': chunk.coverage_reserved,
            'rerank': chunk.rerank_score,
            'url': chunk.url,
        }
        for index, chunk in enumerate(pipeline.last_chunks, start=1)
    ]))

    retrieval_debug = pd.DataFrame(pipeline.retriever.last_debug_rows)
    debug_columns = [name for name in [
        'title', 'url', 'bm25_rank', 'mmr_rank', 'rerank_rank',
        'rerank_score', 'selection_score', 'anchor_candidate',
        'anchor_title_candidate', 'coverage_reserved', 'final_rank',
    ] if name in retrieval_debug]
    display(
        retrieval_debug[debug_columns]
        .sort_values(['final_rank', 'selection_score'], ascending=[True, False], na_position='last')
        .head(50)
    )


,valeur
retrieved_chunks,10
included_chunks,10
context_tokens,2718
prompt_tokens,4550
input_token_limit,4608
reserved_output_tokens,1536
answer_mode,enumeration
membership_validation,False
coverage_required_chunks,6
query_focused_passages,10


,index,passage,url,title,section,similarity_score,rerank_score
0,1,# cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquen...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,0.693846,0.990404
1,2,# cdr.ensea.1 — 🎤✨ Le CEE vous présente son qu...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,cdr.ensea.1 — 🎤✨ Le CEE vous présente son quat...,cdr.ensea.1 — 🎤✨ Le CEE vous présente son quat...,0.612673,0.989592
2,3,# cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquen...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,0.659445,0.988085
3,4,# cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquen...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,0.641377,0.984516
4,5,# cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquen...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,0.639711,0.983971
5,6,# cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquen...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,0.670147,0.982757
6,7,# cdr.ensea.1 — 🎤✨ Revivons ensemble une final...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,cdr.ensea.1 — 🎤✨ Revivons ensemble une finale ...,cdr.ensea.1 — 🎤✨ Revivons ensemble une finale ...,0.649620,0.995882
7,8,## Texte detecte dans les images - ENSEA CDR E...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,cdr.ensea.1 — 🎤✨ L’avenir du CDR se dessine !,cdr.ensea.1 — 🎤✨ L’avenir du CDR se dessine ! ...,0.578844,0.994509
8,9,# cdr.ensea.1 — 🎤✨La présélection du Concours ...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,cdr.ensea.1 — 🎤✨La présélection du Concours EN...,cdr.ensea.1 — 🎤✨La présélection du Concours EN...,0.596845,0.990102
9,10,# cdr.ensea.1 — 🎤✨ Une demi-finale enflammée s...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,cdr.ensea.1 — 🎤✨ Une demi-finale enflammée s’e...,cdr.ensea.1 — 🎤✨ Une demi-finale enflammée s’e...,0.615161,0.988842


,rang,titre,dans_le_prompt,couverture_obligatoire,rerank,url
0,1,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,True,True,0.990404,https://www.facebook.com/cdr.ensea.1/posts/pfb...
1,2,cdr.ensea.1 — 🎤✨ Le CEE vous présente son quat...,True,True,0.989592,https://www.facebook.com/cdr.ensea.1/posts/pfb...
2,3,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,True,True,0.988085,https://www.facebook.com/cdr.ensea.1/posts/pfb...
3,4,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,True,True,0.984516,https://www.facebook.com/cdr.ensea.1/posts/pfb...
4,5,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,True,True,0.983971,https://www.facebook.com/cdr.ensea.1/posts/pfb...
5,6,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,True,True,0.982757,https://www.facebook.com/cdr.ensea.1/posts/pfb...
6,7,cdr.ensea.1 — 🎤✨ Revivons ensemble une finale ...,True,False,0.995882,https://www.facebook.com/cdr.ensea.1/posts/pfb...
7,8,cdr.ensea.1 — 🎤✨ L’avenir du CDR se dessine !,True,False,0.994509,https://www.facebook.com/cdr.ensea.1/posts/pfb...
8,9,cdr.ensea.1 — 🎤✨La présélection du Concours EN...,True,False,0.990102,https://www.facebook.com/cdr.ensea.1/posts/pfb...
9,10,cdr.ensea.1 — 🎤✨ Une demi-finale enflammée s’e...,True,False,0.988842,https://www.facebook.com/cdr.ensea.1/posts/pfb...


,title,url,bm25_rank,mmr_rank,rerank_rank,rerank_score,selection_score,anchor_candidate,anchor_title_candidate,coverage_reserved,final_rank
1,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,5,2,1,0.990404,1.140404,True,True,True,1.0
6,cdr.ensea.1 — 🎤✨ Le CEE vous présente son quat...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,4,7,2,0.989592,1.139592,True,True,True,2.0
4,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,6,5,3,0.988085,1.138085,True,True,True,3.0
2,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,1,3,4,0.984516,1.134516,True,True,True,4.0
3,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,3,4,5,0.983971,1.133971,True,True,True,5.0
0,cdr.ensea.1 — 🎤✨ Le Concours ENSEA d’Éloquence...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,2,1,6,0.982757,1.132757,True,True,True,6.0
5,cdr.ensea.1 — 🎤✨ Revivons ensemble une finale ...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,7,6,7,0.995882,0.995882,False,False,False,7.0
27,cdr.ensea.1 — 🎤✨ L’avenir du CDR se dessine !,https://www.facebook.com/cdr.ensea.1/posts/pfb...,31,28,8,0.994509,0.994509,False,False,False,8.0
15,cdr.ensea.1 — 🎤✨La présélection du Concours EN...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,10,16,10,0.990102,0.990102,False,False,False,9.0
13,cdr.ensea.1 — 🎤✨ Une demi-finale enflammée s’e...,https://www.facebook.com/cdr.ensea.1/posts/pfb...,16,15,11,0.988842,0.988842,False,False,False,10.0


## Cache exact

La clé du cache contient la question normalisée, les chunks, l'historique compact, le prompt et les paramètres du modèle. Une nouvelle politique de réponse ou une conversation différente ne peut donc pas recevoir une ancienne réponse incompatible.
